In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
)

# Load data
data = np.loadtxt(r"C:\Users\Sam\Desktop\ML\data\Data_err.npt")
y_real = data[:, 0]
y_pred = data[:, 1]

# Split into train/test
split_idx = int(len(y_real) * 0.8)
y_real_train, y_real_test = y_real[:split_idx], y_real[split_idx:]
y_pred_train, y_pred_test = y_pred[:split_idx], y_pred[split_idx:]

# Metric function
def get_metrics(y_true, y_pred):
    return {
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }

# Compute metrics
metrics_all = get_metrics(y_real, y_pred)
metrics_train = get_metrics(y_real_train, y_pred_train)
metrics_test = get_metrics(y_real_test, y_pred_test)

# Create main metrics DataFrame
df_main = pd.DataFrame(
    [
        ["All", *metrics_all.values()],
        ["Train", *metrics_train.values()],
        ["Test", *metrics_test.values()],
    ],
    columns=["Set", "Recall", "Accuracy", "F1", "Precision", "MCC"],
)

# Compute per-class metrics using average=None
precision_per_class = precision_score(y_real, y_pred, average=None, zero_division=0)
recall_per_class = recall_score(y_real, y_pred, average=None, zero_division=0)
f1_per_class = f1_score(y_real, y_pred, average=None, zero_division=0)

# Accuracy per class: proportion of correct predictions for each class
accuracy_per_class = []
for cls in np.unique(y_real):
    idx = y_real == cls
    acc = accuracy_score(y_real[idx], y_pred[idx])
    accuracy_per_class.append(acc)

# Build DataFrame
df_class = pd.DataFrame(
    {
        "Class": np.unique(y_real).astype(int),
        "Recall": recall_per_class,
        "Accuracy": accuracy_per_class,
        "F1": f1_per_class,
        "Precision": precision_per_class,
    }
)

# Display both tables
# print("Main Metrics Table:")
# print(df_main.to_string(index=False))
# print("\nPer-Class Metrics Table:")
# print(df_class.to_string(index=False))

# ROC and AUC
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_real, y_pred)
roc_auc = auc(fpr, tpr)
auc_column = [""] * (len(fpr) - 1) + [round(roc_auc, 3)]
roc_df = pd.DataFrame({"FPR": fpr, "TPR": tpr, "AUC": auc_column})
# print("\nROC Curve Data:")
# print(roc_df)

# Confusion Matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_real, y_pred)
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
# print("\nConfusion Matrix:")
# print(cm_df)

# Plot heatmap
# sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
# plt.title("Confusion Matrix Heatmap")
# plt.show()

In [7]:
df_main.to_clipboard(index=False)

In [8]:
df_class.to_clipboard(index=False)

In [4]:
roc_df.to_clipboard(index=False)

In [5]:
cm_df.to_clipboard(index=False)

In [6]:
import pandas as pd
import numpy as np
import smogn

# -------------------- 1. Load your dataset --------------------
file_path = r"C:\Users\Sam\Desktop\ML\task\BSS.No.1-Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="Z-Score")

# -------------------- 2. Define CPU usage column --------------------
cpu_col = "CPU Usage"  # Replace with your actual column name

# -------------------- 3. Bin CPU usage into Low, Medium, High --------------------
bins = [0, 20, 60, 100]
labels = ["Low", "Medium", "High"]
df["CPU_Bin"] = pd.cut(df[cpu_col], bins=bins, labels=labels, include_lowest=True)

# -------------------- 4. Analyze bin distribution --------------------
bin_counts = df["CPU_Bin"].value_counts().sort_index()
bin_percentages = bin_counts / len(df) * 100

print("CPU Usage Distribution:")

print(pd.DataFrame({"Count": bin_counts, "Percentage": bin_percentages.round(2)}))

# -------------------- 5. Check for imbalance --------------------
threshold = 10  # percent
rare_bins = bin_percentages[bin_percentages < threshold]

# -------------------- 6. Apply SMOGN if imbalance is detected --------------------
if not rare_bins.empty:
    print("\n⚠️ Imbalance detected in bins:", list(rare_bins.index))
    print("→ Applying SMOGN to balance rare CPU usage ranges...")

    df_smogn = smogn.smoter(
        data=df.drop(columns=["CPU_Bin"]),
        y=cpu_col,
        k=5,
        samp_method="extreme",
        rel_thres=0.8,
        rel_method="auto",
        under_samp=True
    )
else:
    print("\n✅ No imbalance detected. Skipping SMOGN.")
    df_smogn = df.drop(columns=["CPU_Bin"])

# -------------------- 7. Save to clipboad --------------------
df_smogn.to_clipboard(index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Sam\\Desktop\\ML\\task\\BSS.No.1-Dataset.xlsx'

In [ ]:
pd.DataFrame({"Count": bin_counts, "Percentage": bin_percentages.round(2)}).to_clipboard()